# 02 - Cleaning & Feature Engineering
- Input: `data/raw.csv`
- Output: `data/processed.csv` + `fmcg_clean.parquet`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw.csv') 
if not RAW.exists():
    RAW = Path('data/raw.csv')
OUT = Path('../data'); OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW, parse_dates=['date'])
print('Raw:', df.shape)

## 1. Drop corrupt rows (negative quantities) — logged

In [ ]:
neg_mask = (df['stock_available'] < 0) | (df['delivered_qty'] < 0) | (df['units_sold'] < 0)
removed = df[neg_mask].copy()
print('Rows to drop:', neg_mask.sum())
display(removed[['date','sku','channel','region','stock_available','delivered_qty','units_sold']])
removed.to_csv(OUT / 'removed_negative_rows.csv', index=False)

df = df[~neg_mask].reset_index(drop=True)
print('After drop:', df.shape)

## 2. Sort & type hygiene

In [ ]:
df = df.sort_values(['sku','region','channel','date']).reset_index(drop=True)
cat_cols = ['sku','brand','segment','category','channel','region','pack_type']
for c in cat_cols:
    df[c] = df[c].astype('category')
df['promotion_flag'] = df['promotion_flag'].astype('int8')
print(df.dtypes)

## 3. Feature engineering

In [ ]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['week'] = df['date'].dt.isocalendar().week.astype('int16')
df['year_week'] = df['date'].dt.strftime('%G-W%V')
df['dow'] = df['date'].dt.dayofweek            # 0=Mon
df['is_weekend'] = (df['dow'] >= 5).astype('int8')
df['quarter'] = df['date'].dt.quarter

df['revenue_proxy'] = (df['price_unit'] * df['units_sold']).round(2)

df['is_stockout']  = (df['stock_available'] == 0).astype('int8')
df['is_zero_sales'] = (df['units_sold'] == 0).astype('int8')

df['sell_through'] = np.where(df['stock_available'] > 0, (df['units_sold'] / df['stock_available']).round(4), np.nan)
df.head()

## 4. Post-cleaning sanity checks

In [ ]:
checks = {
    'no negatives': int(((df[['stock_available','delivered_qty','units_sold','price_unit']] < 0).any(axis=1)).sum()),
    'no missing in key cols': int(df[['date','sku','units_sold']].isna().sum().sum()),
    'rows': len(df),
    'date min': str(df['date'].min().date()),
    'date max': str(df['date'].max().date()),
    'skus': df['sku'].nunique(),
    'stockout rows': int(df['is_stockout'].sum()),
}
for k,v in checks.items():
    print(f'{k:24s}: {v}')
assert checks['no negatives'] == 0, 'Negatives remain!'
assert len(df) == 190757 - 3, 'Row count unexpected'
print('\nAll assertions passed.')

## 5. Export cleaned dataset

In [ ]:
df.to_csv(OUT / 'processed.csv', index=False)
try:
    df.to_parquet(OUT / 'fmcg_clean.parquet', index=False)
    print('Wrote csv + parquet to', OUT.resolve())
except Exception as e:
    print('CSV written. Parquet skipped (install pyarrow):', e)
print('Final shape:', df.shape)
print('Columns:', list(df.columns))